## Task 1: Multi-Agent Design Thinking

### Role1: Competitor Researcher
- Role: Competitor Researcher
- Goal: Collect accurate competitor facts and organize them into an evidence brief.
- Backstory: A careful research analyst who separates observed facts from interpretation and never invents missing data.
- Tool access: competitor lookup only.

### Role2: Strategy Analyst
- Role: Strategy Analyst
- Goal: Turn evidence into defensible competitive positioning and opportunities.
- Backstory: A business strategist who focuses on differentiation, trade-offs, and evidence-backed decisions.
- Tool access: price comparison only.

### Role3: Marketing Writer
- Role: Marketing Writer
- Goal: Produce a concise, useful recommendation grounded in the research and strategy.
- Backstory: A practical marketing writer who values clarity, factual grounding, and stakeholder usefulness over hype.
- Tool access: marketing-claim validation only.

### Why specialists can outperform one generalist:
Specialists can keep responsibilities narrow, give each agent only the tools it needs, and make handoffs explicit. A generalist is often better for a small, simple task where coordination overhead would cost more than the extra specialization.

In [1]:
# Install the notebook dependencies.
%pip install -q -U crewai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.9/198.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.

In [2]:
import os
import json
import time
from getpass import getpass

import pandas as pd
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

MODEL = "gemini/gemini-3.6-flash"

# Four separate keys: one key per specialist/manager role
API_KEY_1 = getpass("Enter Gemini API Key 1 (Researcher): ")
API_KEY_2 = getpass("Enter Gemini API Key 2 (Strategy Analyst): ")
API_KEY_3 = getpass("Enter Gemini API Key 3 (Marketing Writer): ")
API_KEY_4 = getpass("Enter Gemini API Key 4 (Hierarchical Manager): ")

research_llm = LLM(model=MODEL, api_key=API_KEY_1, temperature=0)
strategy_llm = LLM(model=MODEL, api_key=API_KEY_2, temperature=0.2)
writer_llm = LLM(model=MODEL, api_key=API_KEY_3, temperature=0.4)
manager_llm = LLM(model=MODEL, api_key=API_KEY_4, temperature=0)

print("CrewAI setup complete: 4 API keys assigned to 4 roles.")

Enter Gemini API Key 1 (Researcher): ··········
Enter Gemini API Key 2 (Strategy Analyst): ··········
Enter Gemini API Key 3 (Marketing Writer): ··········
Enter Gemini API Key 4 (Hierarchical Manager): ··········
CrewAI setup complete: 4 API keys assigned to 4 roles.


## Task 2: Build the Agents and Give Each Role-Appropriate Tools

The tools use a small local dataset, so the assignment does not need a separate search API. Each agent receives only the tool relevant to its role

In [3]:
# Local competitor dataset (reproducible and does not require a search-service API key)
competitors = pd.DataFrame([
    {"company": "AlphaBox", "price": 39, "delivery_days": 2, "subscription": True, "positioning": "low-cost convenience", "rating": 4.1},
    {"company": "FreshCart", "price": 49, "delivery_days": 1, "subscription": True, "positioning": "speed and freshness", "rating": 4.4},
    {"company": "GreenBasket", "price": 59, "delivery_days": 2, "subscription": False, "positioning": "premium organic quality", "rating": 4.6},
])
competitors.to_csv("competitors.csv", index=False)

@tool("lookup_competitor")
def lookup_competitor(company: str) -> str:
    """Return the available competitor facts for a named competitor."""
    row = competitors[competitors["company"].str.lower() == company.lower()]
    if row.empty:
        return f"No competitor named {company} was found."
    return str(row.to_dict(orient="records")[0])

@tool("compare_prices")
def compare_prices(price_a: float, price_b: float) -> str:
    """Calculate absolute and percentage price difference."""
    diff = abs(price_a - price_b)
    pct = (diff / min(price_a, price_b)) * 100 if min(price_a, price_b) else 0
    return f"Difference: ${diff:.2f}; relative difference versus the cheaper offer: {pct:.1f}%"

@tool("validate_marketing_claim")
def validate_marketing_claim(claim: str) -> str:
    """Check a marketing claim against the local competitor facts."""
    lower = claim.lower()
    checks = []
    if "fast" in lower or "same day" in lower:
        checks.append(f"Fastest listed delivery is {competitors['delivery_days'].min()} day(s).")
    if "cheapest" in lower or "low-cost" in lower:
        cheapest = competitors.loc[competitors["price"].idxmin(), "company"]
        checks.append(f"Lowest listed price belongs to {cheapest} at ${competitors['price'].min():.0f}.")
    if not checks:
        checks.append("No direct matching claim was found in the dataset; treat the claim as unverified.")
    return " ".join(checks)

researcher = Agent(
    role="Competitor Researcher",
    goal="Collect accurate competitor facts and present a structured evidence brief.",
    backstory="You are a careful research analyst. Separate observed facts from interpretation and never invent missing data.",
    llm=research_llm,
    tools=[lookup_competitor],
    allow_delegation=False,
    verbose=True,
    max_iter=3,
)

strategist = Agent(
    role="Strategy Analyst",
    goal="Turn research evidence into clear competitive positioning and a defensible marketing angle.",
    backstory="You are a business strategist who looks for differentiation, trade-offs, and evidence-backed opportunities.",
    llm=strategy_llm,
    tools=[compare_prices],
    allow_delegation=False,
    verbose=True,
    max_iter=3,
)

writer = Agent(
    role="Marketing Writer",
    goal="Produce a concise stakeholder-ready recommendation grounded only in the research and strategy.",
    backstory="You are a practical marketing writer. You value clarity, factual grounding, and useful recommendations over hype.",
    llm=writer_llm,
    tools=[validate_marketing_claim],
    allow_delegation=False,
    verbose=True,
    max_iter=3,
)

print("Three specialized agents created with separate API keys and role-specific tools.")

Three specialized agents created with separate API keys and role-specific tools.


## Task 3: CrewAI Task Objects and Sequential Execution

The tasks have explicit descriptions, expected outputs, and context dependencies. This makes the handoffs predictable and also addresses the required output-format issue

In [6]:
import asyncio

research_task = Task(
    description=(
        "Research AlphaBox, FreshCart, and GreenBasket using the competitor lookup tool. "
        "Return exactly three entries. For each entry include: company, price, delivery_days, "
        "subscription, positioning, rating, and one evidence note. Do not invent fields."
    ),
    expected_output=(
        "Exactly three clearly labeled competitor entries. Each entry must contain the fields "
        "company, price, delivery_days, subscription, positioning, rating, and evidence note."
    ),
    agent=researcher,
)

strategy_task = Task(
    description=(
        "Use the research brief to compare the competitors and identify 2-3 defensible opportunities "
        "for a new entrant. Use the price comparison tool where useful. Clearly separate facts from "
        "interpretation and finish with one recommended positioning angle."
    ),
    expected_output=(
        "A strategy brief with: factual observations, 2-3 opportunities, key trade-offs, and exactly "
        "one recommended positioning angle. Facts and interpretation must be clearly separated."
    ),
    agent=strategist,
    context=[research_task],
)

writing_task = Task(
    description=(
        "Use the research and strategy outputs to write a stakeholder-ready marketing recommendation. "
        "Include the target positioning, three supporting points, two risks, and one short sample message. "
        "Use the validation tool before making a factual marketing claim. Keep the answer concise."
    ),
    expected_output=(
        "A polished recommendation containing: target positioning, 3 supporting points, 2 risks, "
        "and 1 short sample message. The sample message must avoid unsupported factual claims."
    ),
    agent=writer,
    context=[research_task, strategy_task],
)

sequential_crew = Crew(
    agents=[researcher, strategist, writer],
    tasks=[research_task, strategy_task, writing_task],
    process=Process.sequential,
    verbose=True,
)

async def run_sequential_crew():
    global sequential_result, sequential_seconds
    start = time.perf_counter()
    sequential_result = await sequential_crew.kickoff_async()
    sequential_seconds = time.perf_counter() - start

    print("\n=== SEQUENTIAL FINAL OUTPUT ===")
    print(sequential_result)
    print(f"\nSequential elapsed time: {sequential_seconds:.2f}s")

await run_sequential_crew()


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7caee6fe-1267-4924-a849-fb12ae304c1c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research AlphaBox, FreshCart, and GreenBasket using the competitor lookup tool. Return exactly three     │
│  entries. For each entry include: company, price, delivery_days, subscription, positioning, rating, and one     │
│  evidence note. Do not invent fields.                                                                           │
│  ID: 9d96a56b-0d99-4d8e-96b2-a9924dc6f614                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Competitor Researcher                                                                                   │
│                                                                                                                 │
│  Task: Research AlphaBox, FreshCart, and GreenBasket using the competitor lookup tool. Return exactly three     │
│  entries. For each entry include: company, price, delivery_days, subscription, positioning, rating, and one     │
│  evidence note. Do not invent fields.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: lookup_competitor                                                                                        │
│  Args: {'company': 'AlphaBox'}                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool lookup_competitor executed with result: {'company': 'AlphaBox', 'price': 39, 'delivery_days': 2, 'subscription': True, 'positioning': 'low-cost convenience', 'rating': 4.1}...


...

Maximum iterations reached. Requesting final answer.
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Marketing Recommendation: Flex-Fresh Positioning Strategy                                                  │
│                                                                                                                 │
│  **Target Positioning**                                                                                         │
│  **"Flex-Fresh: Premium Speed & Quality Without the Lock-In"**                                                  │
│  Position the product as an on-demand, non-subscription grocery delivery service offering 1-day fulfillment at  │
│  a competitive $49 price point.                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Supporting Strategic Points                                                                                │
│                                                                                                                 │
│  1. **Fills a Distinct Market Void:** No current market player combines 1-day delivery speed with a flexible,   │
│  non-subscription purchasing model.                                                                             │
│  2. **Neutralizes Mid-Tier Competitors:** Matches FreshCart’s $49 price point and 1-day delivery speed while    │
│  removing mandatory subscription lock-in to eliminate onboarding friction.                                      │
│  3. **Outperforms Premium Competitors on Speed and Price:** Undercuts GreenBasket ($59) by 17% ($49 vs. $59)    │
│  while delivering orders a full day faster (1 day vs. 2 days).                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Key Risks                                                                                                  │
│                                                                                                                 │
│  1. **Margin Compression from Logistics Spend:** Offering 1-day delivery without requiring a subscription       │
│  increases unit logistics spend and fulfillment complexity.                                                     │
│  2. **Revenue Volatility:** Waiving mandatory subscriptions lowers predictable Monthly Recurring Revenue (MRR)  │
│  and may reduce long-term customer Lifetime Value (LTV), requiring continuous acquisition efforts.              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Sample Message                                    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Use the research and strategy outputs to write a stakeholder-ready marketing recommendation. Include     │
│  the target positioning, three supporting points, two risks, and one short sample message. Use the validation   │
│  tool before making a factual marketing claim. Keep the answer concise.                                         │
│  Agent: Marketing Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== SEQUENTIAL FINAL OUTPUT ===
### Marketing Recommendation: Flex-Fresh Positioning Strategy

**Target Positioning**  
**"Flex-Fresh: Premium Speed & Quality Without the Lock-In"**  
Position the product as an on-demand, non-subscription grocery delivery service offering 1-day fulfillment at a competitive $49 price point. 

---

### Supporting Strategic Points

1. **Fills a Distinct Market Void:** No current market player combines 1-day delivery speed with a flexible, non-subscription purchasing model.
2. **Neutralizes Mid-Tier Competitors:** Matches FreshCart’s $49 price point and 1-day delivery speed while removing mandatory subscription lock-in to eliminate onboarding friction.
3. **Outperforms Premium Competitors on Speed and Price:** Undercuts GreenBasket ($59) by 17% ($49 vs. $59) while delivering orders a full day faster (1 day vs. 2 days).

---

### Key Risks

1. **Margin Compression from Logistics Spend:** Offering 1-day delivery without requiring a subscription increases un

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7caee6fe-1267-4924-a849-fb12ae304c1c                                                                       │
│  Final Output: ### Marketing Recommendation: Flex-Fresh Positioning Strategy                                    │
│                                                                                                                 │
│  **Target Positioning**                                                                                         │
│  **"Flex-Fresh: Premium Speed & Quality Without the Lock-In"**                                                  │
│  Position the product as an on-demand, non-subscription grocery delivery service offering 1-day fulfillment at  │
│  a competitive $49 price point.                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Supporting Strategic Points                                                                                │
│                                                                                                                 │
│  1. **Fills a Distinct Market Void:** No current market player combines 1-day delivery speed with a flexible,   │
│  non-subscription purchasing model.                                                                             │
│  2. **Neutralizes Mid-Tier Competitors:** Matches FreshCart’s $49 price point and 1-day delivery speed while    │
│  removing mandatory subscription lock-in to eliminate onboarding friction.                                      │
│  3. **Outperforms Premium Competitors on Speed and Price:** Undercuts GreenBasket ($59) by 17% ($49 vs. $59)    │
│  while delivering orders a full day faster (1 day vs. 2 days).                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Key Risks                                                                                                  │
│                                                                                                                 │
│  1. **Margin Compression from Logistics Spend:** Offering 1-day delivery without requiring a subscription       │
│  increases unit logistics spend and fulfillment complexity.                                                     │
│  2. **Revenue Volatility:** Waiving mandatory subscriptions lowers predictable Monthly Recurring Revenue (MRR)  │
│  and may reduce long-term customer Lifetime Value (LTV), requiring continuous acquisition efforts.              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Sample Message                                   

In [7]:
# Captures and review all task outputs from the sequential run
def show_task_outputs(result, label):
    print(f"\n=== {label} TASK OUTPUTS ===")
    outputs = getattr(result, "tasks_output", []) or []
    if not outputs:
        print("No per-task output list was exposed by this CrewAI version.")
        return
    for i, output in enumerate(outputs, start=1):
        print(f"\n--- Task {i} ---")
        print(output)

show_task_outputs(sequential_result, "SEQUENTIAL")


=== SEQUENTIAL TASK OUTPUTS ===

--- Task 1 ---
### Competitor Entry 1
* **company**: AlphaBox
* **price**: 39
* **delivery_days**: 2
* **subscription**: True
* **positioning**: low-cost convenience
* **rating**: 4.1
* **evidence note**: Data confirmed via competitor lookup tool for AlphaBox.

---

### Competitor Entry 2
* **company**: FreshCart
* **price**: 49
* **delivery_days**: 1
* **subscription**: True
* **positioning**: speed and freshness
* **rating**: 4.4
* **evidence note**: Data confirmed via competitor lookup tool for FreshCart.

---

### Competitor Entry 3
* **company**: GreenBasket
* **price**: 59
* **delivery_days**: 2
* **subscription**: False
* **positioning**: premium organic quality
* **rating**: 4.6
* **evidence note**: Data confirmed via competitor lookup tool for GreenBasket.

--- Task 2 ---
# Competitive Strategy Brief: New Entrant Market Opportunity & Positioning

---

## 1. Factual Observations vs. Strategic Interpretation

### A. Factual Observations
* **Alph

### Required output-format mismatch and fix

One handoff risk is a mismatch between what the downstream agent expects and what the upstream agent returns. For example, the researcher may initially return a free-form paragraph even though the strategy agent needs named competitor fields. The fix is to make the research task's expected_output explicitly require the exact fields and to repeat the required structure in the task description. The strategy and writing tasks use the same approach, so downstream agents receive stable, predictable handoffs

## Task 4: Hierarchical Process with a Manager Agent

The hierarchical version uses the same specialist agents but adds a dedicated manager using API Key 4. The manager can delegate work, review specialist outputs, and coordinate the final answer

In [9]:
manager = Agent(
    role="Crew Manager",
    goal="Delegate competitor-analysis work to the right specialists, review their outputs, resolve gaps, and deliver a coherent final recommendation.",
    backstory="You are an experienced project manager. Assign work according to specialization, check deliverables, and request corrections when needed.",
    llm=manager_llm,
    allow_delegation=True,
    verbose=True,
    max_iter=4,
)

hierarchical_task = Task(
    description=(
        "Complete the competitor-analysis objective. Delegate research to the Competitor Researcher, "
        "strategy work to to the Strategy Analyst, and final recommendation writing to the Marketing Writer. "
        "Review the specialist work before finishing. The final response must contain evidence, "
        "2-3 opportunities, one positioning angle, 2 risks, and one short sample message."
    ),
    expected_output=(
        "A final stakeholder-ready competitor recommendation with a concise evidence summary, "
        "2-3 opportunities, one recommended positioning angle, 2 risks, and one short sample message."
    ),
)

hierarchical_crew = Crew(
    agents=[researcher, strategist, writer],
    tasks=[hierarchical_task],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True,
)

async def run_hierarchical_crew():
    global hierarchical_result, hierarchical_seconds
    start = time.perf_counter()
    hierarchical_result = await hierarchical_crew.kickoff_async()
    hierarchical_seconds = time.perf_counter() - start

    print("\n=== HIERARCHICAL FINAL OUTPUT ===")
    print(hierarchical_result)
    print(f"\nHierarchical elapsed time: {hierarchical_seconds:.2f}s")

await run_hierarchical_crew()


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5a2fbcd4-e057-4b7a-9140-844730cdbe4f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Complete the competitor-analysis objective. Delegate research to the Competitor Researcher, strategy     │
│  work to to the Strategy Analyst, and final recommendation writing to the Marketing Writer. Review the          │
│  specialist work before finishing. The final response must contain evidence, 2-3 opportunities, one             │
│  positioning angle, 2 risks, and one short sample message.                                                      │
│  ID: 4762c2cb-75aa-4ddf-bc33-b95eedba9606                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Complete the competitor-analysis objective. Delegate research to the Competitor Researcher, strategy     │
│  work to to the Strategy Analyst, and final recommendation writing to the Marketing Writer. Review the          │
│  specialist work before finishing. The final response must contain evidence, 2-3 opportunities, one             │
│  positioning angle, 2 risks, and one short sample message.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': 'We are conducting a comprehensive competitor analysis for our mid-market B2B AI Workflow &  │
│  Productivity SaaS platform. We need an up-to-date, detailed competitor research report analyzing...            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Competitor Researcher                                                                                   │
│                                                                                                                 │
│  Task: Conduct competitor research on top B2B AI Productivity & Workflow Automation competitors, identifying    │
│  key strengths, weaknesses, customer pain points, positioning, and market evidence.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Competitor Recommendation & Strategy Report: Mid-Market AI Work System                                       │
│                                                                                                                 │
│  **To:** Executive Leadership & Product Strategy Teams                                                          │
│  **From:** Crew Lead / Project Manager                                                                          │
│  **Subject:** Competitor Analysis, Strategic Positioning, and Go-to-Market Recommendation                       │
│  **Target Market:** Mid-Market B2B AI Productivity & Workflow Automation SaaS (100–1,000 seats)                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Executive Evidence Summary                                                                               │
│                                                                                                                 │
│  Analysis of primary mid-market competitors reveals critical gaps in execution capabilities, enterprise         │
│  governance, and pricing structures:                                                                            │
│                                                                                                                 │
│  *   **Notion AI:** Strong in unstructured document creation and global knowledge search. However, it lacks     │
│  native multi-condition workflow automation, experiences severe database latency at scale (100+ seats), and     │
│  imposes an **$8–$10/user/month add-on tax** on top of paid base subscriptions.                                 │
│  *   **Asana Intelligence:** Delivers strong project health reporting and executive status generation, but      │
│  gates AI features behind its **$24.99/user/month Advanced tier** (a **149.9% price premium** over standard     │
│  plans). It lacks native document authoring and relies on rigid, non-adaptive custom workflows.                 │
│  *   **ClickUp Brain:** Offers high feature density across docs and tasks at a lower price point ($7/user/mo    │
│  add-on), but is severely hindered by **UI bloat, platform latency, steep learning curves**, and overly         │
│  complex permission hierarchies that trigger deployment delays in mid-market organizations.                     │
│  *   **Core Cross-Market Pain Points:**                                                                         │
│      1.  **Procurement Friction:** Buyers face forced tier migrations and unpredictable budget expansion        │
│  caused by per-seat AI surcharges.                                                                              │
│      2.  **IT & Compliance Roadblocks:** Security teams regularly block adoption due to ungoverned LLM          │
│  indexing across sensitive company data without granular boundary controls.                                     │
│                                                                                                                 │
│  ---                                                   

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Complete the competitor-analysis objective. Delegate research to the Competitor Researcher, strategy     │
│  work to to the Strategy Analyst, and final recommendation writing to the Marketing Writer. Review the          │
│  specialist work before finishing. The final response must contain evidence, 2-3 opportunities, one             │
│  positioning angle, 2 risks, and one short sample message.                                                      │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== HIERARCHICAL FINAL OUTPUT ===
# Competitor Recommendation & Strategy Report: Mid-Market AI Work System

**To:** Executive Leadership & Product Strategy Teams  
**From:** Crew Lead / Project Manager  
**Subject:** Competitor Analysis, Strategic Positioning, and Go-to-Market Recommendation  
**Target Market:** Mid-Market B2B AI Productivity & Workflow Automation SaaS (100–1,000 seats)

---

## 1. Executive Evidence Summary

Analysis of primary mid-market competitors reveals critical gaps in execution capabilities, enterprise governance, and pricing structures:

*   **Notion AI:** Strong in unstructured document creation and global knowledge search. However, it lacks native multi-condition workflow automation, experiences severe database latency at scale (100+ seats), and imposes an **$8–$10/user/month add-on tax** on top of paid base subscriptions.
*   **Asana Intelligence:** Delivers strong project health reporting and executive status generation, but gates AI features behind its *

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5a2fbcd4-e057-4b7a-9140-844730cdbe4f                                                                       │
│  Final Output: # Competitor Recommendation & Strategy Report: Mid-Market AI Work System                         │
│                                                                                                                 │
│  **To:** Executive Leadership & Product Strategy Teams                                                          │
│  **From:** Crew Lead / Project Manager                                                                          │
│  **Subject:** Competitor Analysis, Strategic Positioning, and Go-to-Market Recommendation                       │
│  **Target Market:** Mid-Market B2B AI Productivity & Workflow Automation SaaS (100–1,000 seats)                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Executive Evidence Summary                                                                               │
│                                                                                                                 │
│  Analysis of primary mid-market competitors reveals critical gaps in execution capabilities, enterprise         │
│  governance, and pricing structures:                                                                            │
│                                                                                                                 │
│  *   **Notion AI:** Strong in unstructured document creation and global knowledge search. However, it lacks     │
│  native multi-condition workflow automation, experiences severe database latency at scale (100+ seats), and     │
│  imposes an **$8–$10/user/month add-on tax** on top of paid base subscriptions.                                 │
│  *   **Asana Intelligence:** Delivers strong project health reporting and executive status generation, but      │
│  gates AI features behind its **$24.99/user/month Advanced tier** (a **149.9% price premium** over standard     │
│  plans). It lacks native document authoring and relies on rigid, non-adaptive custom workflows.                 │
│  *   **ClickUp Brain:** Offers high feature density across docs and tasks at a lower price point ($7/user/mo    │
│  add-on), but is severely hindered by **UI bloat, platform latency, steep learning curves**, and overly         │
│  complex permission hierarchies that trigger deployment delays in mid-market organizations.                     │
│  *   **Core Cross-Market Pain Points:**                                                                         │
│      1.  **Procurement Friction:** Buyers face forced tier migrations and unpredictable budget expansion        │
│  caused by per-seat AI surcharges.                                                                              │
│      2.  **IT & Compliance Roadblocks:** Security teams regularly block adoption due to ungoverned LLM          │
│  indexing across sensitive company data without granular boundary controls.                                     │
│                                                                                                                 │
│  ---                                                  

In [10]:
show_task_outputs(hierarchical_result, "HIERARCHICAL")


=== HIERARCHICAL TASK OUTPUTS ===

--- Task 1 ---
# Competitor Recommendation & Strategy Report: Mid-Market AI Work System

**To:** Executive Leadership & Product Strategy Teams  
**From:** Crew Lead / Project Manager  
**Subject:** Competitor Analysis, Strategic Positioning, and Go-to-Market Recommendation  
**Target Market:** Mid-Market B2B AI Productivity & Workflow Automation SaaS (100–1,000 seats)

---

## 1. Executive Evidence Summary

Analysis of primary mid-market competitors reveals critical gaps in execution capabilities, enterprise governance, and pricing structures:

*   **Notion AI:** Strong in unstructured document creation and global knowledge search. However, it lacks native multi-condition workflow automation, experiences severe database latency at scale (100+ seats), and imposes an **$8–$10/user/month add-on tax** on top of paid base subscriptions.
*   **Asana Intelligence:** Delivers strong project health reporting and executive status generation, but gates AI featu

### Sequential vs. Hierarchical comparison

| Process | Quality | Latency | Token/cost | Reliability | Best use |
|---|---|---|---|---|---|
| Sequential | Usually strong when the handoff order is known | Usually lower because the path is fixed | Usually lower because coordination is limited | Predictable and easy to debug | Fixed research → analysis → writing pipelines |
| Hierarchical | Can improve coordination and review on open-ended work | Usually higher because the manager adds coordination calls | Usually higher because of manager/delegation overhead | More flexible, but more moving parts | Problems where delegation and review are part of the task |


## Task 5: Token Usage, Approximate Cost, and Success Criteria

CrewAI/provider versions expose usage fields differently, so the helper below checks common locations without assuming one exact result-object shape. Run both crews first, then use the measured values in the comparison table

In [11]:
def usage_dict(result):
    usage = getattr(result, "token_usage", None)
    if usage is None:
        usage = getattr(result, "usage", None)
    if usage is None:
        return {"prompt_tokens": None, "completion_tokens": None, "total_tokens": None}

    def get(*names):
        for name in names:
            if isinstance(usage, dict) and usage.get(name) is not None:
                return usage.get(name)
            value = getattr(usage, name, None)
            if value is not None:
                return value
        return None

    return {
        "prompt_tokens": get("prompt_tokens", "input_tokens"),
        "completion_tokens": get("completion_tokens", "output_tokens"),
        "total_tokens": get("total_tokens"),
    }

seq_usage = usage_dict(sequential_result)
hier_usage = usage_dict(hierarchical_result)

print("Sequential token usage:", seq_usage)
print("Hierarchical token usage:", hier_usage)

# Leaving the current Gemini prices here at 0.0 since i only need token counts and not a price calculation
INPUT_PRICE_PER_MILLION = 0.0
OUTPUT_PRICE_PER_MILLION = 0.0

def approx_cost(usage):
    if usage["prompt_tokens"] is None or usage["completion_tokens"] is None:
        return None
    return (
        usage["prompt_tokens"] / 1_000_000 * INPUT_PRICE_PER_MILLION
        + usage["completion_tokens"] / 1_000_000 * OUTPUT_PRICE_PER_MILLION
    )

seq_cost = approx_cost(seq_usage)
hier_cost = approx_cost(hier_usage)
print("Approx sequential cost:", seq_cost)
print("Approx hierarchical cost:", hier_cost)

Sequential token usage: {'prompt_tokens': 11515, 'completion_tokens': 4386, 'total_tokens': 15901}
Hierarchical token usage: {'prompt_tokens': 38138, 'completion_tokens': 18310, 'total_tokens': 56448}
Approx sequential cost: 0.0
Approx hierarchical cost: 0.0


In [12]:
# Required comparison table using the actual measured results
comparison = pd.DataFrame([
    {
        "process": "Sequential",
        "quality_score_1_to_5": None,
        "latency_seconds": round(sequential_seconds, 2),
        "prompt_tokens": seq_usage["prompt_tokens"],
        "completion_tokens": seq_usage["completion_tokens"],
        "total_tokens": seq_usage["total_tokens"],
        "approx_cost_usd": seq_cost,
        "reliability_notes": "Fixed order and explicit context dependencies",
    },
    {
        "process": "Hierarchical",
        "quality_score_1_to_5": None,
        "latency_seconds": round(hierarchical_seconds, 2),
        "prompt_tokens": hier_usage["prompt_tokens"],
        "completion_tokens": hier_usage["completion_tokens"],
        "total_tokens": hier_usage["total_tokens"],
        "approx_cost_usd": hier_cost,
        "reliability_notes": "Manager delegation and review add flexibility but also coordination overhead",
    },
])
print(comparison.to_string(index=False))

     process quality_score_1_to_5  latency_seconds  prompt_tokens  completion_tokens  total_tokens  approx_cost_usd                                                            reliability_notes
  Sequential                 None            72.66          11515               4386         15901              0.0                                Fixed order and explicit context dependencies
Hierarchical                 None           143.43          38138              18310         56448              0.0 Manager delegation and review add flexibility but also coordination overhead


### Three success criteria

1. **Factual grounding**: recommendations match the supplied competitor evidence and do not invent facts.
2. **Completeness**: the output contains the required positioning, supporting points, risks, and sample message.
3. **Usefulness and clarity**: the recommendation is concise, professional, and actionable for a stakeholder

In [13]:
evaluation = pd.DataFrame([
    {"run": 1, "process": "Sequential", "factual_grounding": None, "completeness": None, "usefulness_clarity": None},
    {"run": 2, "process": "Sequential", "factual_grounding": None, "completeness": None, "usefulness_clarity": None},
    {"run": 3, "process": "Hierarchical", "factual_grounding": None, "completeness": None, "usefulness_clarity": None},
])

evaluation["average"] = evaluation[["factual_grounding", "completeness", "usefulness_clarity"]].mean(axis=1)
print(evaluation)

print("\nSuggested scoring: 5 = excellent, 4 = strong, 3 = acceptable, 2 = weak, 1 = poor.")

   run       process factual_grounding completeness usefulness_clarity average
0    1    Sequential              None         None               None     NaN
1    2    Sequential              None         None               None     NaN
2    3  Hierarchical              None         None               None     NaN

Suggested scoring: 5 = excellent, 4 = strong, 3 = acceptable, 2 = weak, 1 = poor.


### Day 3 LangGraph comparison

Day 3 used an explicit graph state, conditional retry routing, human approval, persistence, and state history/time-travel. CrewAI emphasizes role specialization, task delegation, and manager-led collaboration.

For this same competitor-analysis problem, LangGraph is stronger when precise state transitions, retries, interrupts, and persistence are the main requirement. CrewAI is convenient when the main requirement is dividing work among specialist roles. A useful comparison is therefore: LangGraph gives finer workflow control, while CrewAI gives a higher-level multi-agent collaboration abstraction

## 3-4 sentence conclusion

CrewAI made the competitor-analysis workflow easy to divide into three specialist roles with explicit tool access and task dependencies. The sequential process is more predictable and usually cheaper, while the hierarchical process adds manager-driven delegation and review at the cost of extra coordination. The measured latency and token counts in this notebook should be used to support the final comparison rather than relying only on assumptions. Compared with the Day 3 LangGraph workflow, CrewAI is more focused on agent roles and delegation, while LangGraph provides more direct control over state, routing, persistence, and human-in-the-loop behavior

## Short write up of sequential and hierarchical comparison:

The **sequential CrewAI** process was simpler and more predictable because the three agents worked in a fixed order: the Competitor Researcher gathered information, the Strategy Analyst used that research, and the Marketing Writer turned the strategy into the final output. This made the workflow easy to follow and generally reduced coordination overhead, which is useful for a straightforward pipeline.

The **hierarchical process** added a manager agent that coordinated and delegated work between the specialists. This provides more flexibility for complex tasks because the manager can decide how the agents should collaborate, but it also introduces another reasoning layer. As a result, hierarchical execution can require more tokens and take longer than the sequential approach.

Overall, sequential execution is preferable when the workflow is clearly defined and efficiency is important, while hierarchical execution is more suitable when dynamic delegation and coordination are valuable. The final notebook records the execution metrics so quality, latency, token usage, and reliability can be compared using actual runs.